# STAGE 1 — PRETRAINING (RAW LAW TEXT)

In [1]:
!pip install transformers accelerate bitsandbytes pyarrow pandas
from transformers import BitsAndBytesConfig

from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [ ]:
# dharm3059@gmail.com account
# import os

# BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/LegalAI/LegalAIDatasets/1. textdata"

# CSV_1 = f"{BASE_DIR}/1.2df_txt_norm.csv"
# CSV_2 = f"{BASE_DIR}/2.2df_CSV_norm.csv"
# PARQUET_DIR = f"{BASE_DIR}/3.2_clean_chunks"

In [ ]:
# dpatel11@ltu.edu account

# BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/1. textdata"

# CSV_1 = f"{BASE_DIR}/1.2df_txt_norm.csv"
# CSV_2 = f"{BASE_DIR}/2.2df_CSV_norm.csv"
# PARQUET_DIR = f"{BASE_DIR}/3.2_clean_chunks"

In [2]:
# hetpatel81056 account

BASE_DIR = "/content/drive/MyDrive/LegalAI/LegalAIDatasets/1. textdata"

CSV_1 = f"{BASE_DIR}/1.2df_txt_norm.csv"
CSV_2 = f"{BASE_DIR}/2.2df_CSV_norm.csv"
PARQUET_DIR = f"{BASE_DIR}/3.2_clean_chunks"

In [3]:
import pandas as pd
df1 = pd.read_csv(CSV_1)
df1.head()

,id,text,source,media_type
0,txt_0,=== usc03@119-73.pdf === 401 Extension of Cert...,txt,text
1,txt_1,THE PRESIDENT This title was enacted by act Ju...,txt,text
2,txt_2,CHAPTER 1,txt,text
3,txt_3,—PRESIDENTIAL ELECTIONS AND VACANCIES [Release...,txt,text
4,txt_4,CHAPTER 2,txt,text


In [4]:
import json
import pandas as pd
import pyarrow.parquet as pq
import hashlib
import torch
import random
import os

from torch.utils.data import IterableDataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.optim import AdamW

#Qwen model

In [ ]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

model.gradient_checkpointing_enable()
model.config.use_cache = False

# Option A: reuse EOS as PAD
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

#Streaming data

In [17]:
def stream_csv(path):
    print(f"\n== NOW PROCESSING CSV: {os.path.basename(path)} ==\n")
    for chunk in pd.read_csv(path, chunksize=1000):
      text_col = chunk.columns[1]

      for text in chunk[text_col].dropna(): #Remove empty rows
          yield str(text)    #yield text one by one (streaming)

def stream_parquet(folder):

    files = [f for f in os.listdir(folder) if f.endswith(".parquet")]
    # random.shuffle(files)

    for f in files:
      print(f"\n==NOW PROCESSING: {f} ==\n")
      path = os.path.join(folder, f)

      pf = pq.ParquetFile(path)
      for batch in pf.iter_batches():
        df = batch.to_pandas()

        text_col = df.columns[1]

        for text in df[text_col].dropna():
          yield str(text)

# UNIFIED Stream: Combine all datasets

In [7]:
def unified_stream():
    # CSV 1
    for t in stream_csv(CSV_1):
        yield t

    # CSV 2
    for t in stream_csv(CSV_2):
        yield t

    # Parquet folder
    for t in stream_parquet(PARQUET_DIR):
        yield t

#TRAIN / VAL / TEST SPLIT

In [8]:
def split_filter(text, mode):
    h = int(hashlib.md5(text.encode()).hexdigest(), 16) % 10

    if mode == "train" and h < 8:
        return True
    if mode == "val" and h == 8:
        return True
    if mode == "test" and h == 9:
        return True

    return False

# Chunking

In [9]:
def chunk_tokens(text, max_len=1024, stride=924):
    tokens = tokenizer(text, add_special_tokens=False)["input_ids"]

    for i in range(0, len(tokens), stride): #Moves window forward by 924 tokens.
        chunk = tokens[i:i+max_len] #Takes up to 1024 tokens.

        if len(chunk) > 10:  #Ignores tiny chunks.
            yield chunk

# STAGE 1 DATASET (PRETRAINING)

In [10]:
class Stage1Dataset(IterableDataset):
    def __init__(self, mode):
        self.mode = mode  #Mode = train / val / test

    def __iter__(self):

        for text in unified_stream():

            if not split_filter(text, self.mode):
                continue

            for chunk in chunk_tokens(text):  #Convert tokens → tensor
                yield torch.tensor(chunk)

# collate stage 1

In [11]:
import torch

def collate_stage1(batch, allowed_max_length=None, device="cpu"):
    batch_max_length = max(len(x) + 1 for x in batch)             # Why +1? Because later you: add an EOS token

    input_ids_list, labels_list, attention_mask_list = [], [], [] # input_ids: model inputs labels: targets for loss attention_mask: which tokens are real vs padding

    for x in batch:
        x = x.tolist() if torch.is_tensor(x) else list(x)

        # add EOS
        x = x + [tokenizer.eos_token_id]

        # pad using EOS (Option A rule)
        padded = x + [tokenizer.pad_token_id] * (batch_max_length - len(x))

        # shift for causal LM
        input_ids = torch.tensor(padded[:-1]) #[10, 20, 30, EOS, PAD]
        labels = torch.tensor(padded[1:]) #[20, 30, EOS, PAD, PAD]

        # attention mask: 1 = real token, 0 = padding
        attention_mask = (input_ids != tokenizer.pad_token_id).long()

        # IMPORTANT:
        # mask loss using POSITION, not token ID
        labels = labels.masked_fill(attention_mask == 0, -100)

        # optional truncation
        if allowed_max_length is not None:
            input_ids = input_ids[:allowed_max_length]
            labels = labels[:allowed_max_length]
            attention_mask = attention_mask[:allowed_max_length]

        input_ids_list.append(input_ids)
        labels_list.append(labels)
        attention_mask_list.append(attention_mask)

    return {
    "input_ids": torch.stack(input_ids_list),
    "labels": torch.stack(labels_list),
    "attention_mask": torch.stack(attention_mask_list),
}

In [12]:
ckpt1 = "/content/drive/MyDrive/LegalAI/LegalAIDatasets/qwen_stage1_pretraining_ckpt"
os.makedirs(ckpt1, exist_ok=True)

#Train Stage 1

In [18]:
import os
import re
import torch

# 🔥 NEW: helper to find latest checkpoint
def get_latest_checkpoint(ckpt_dir):
    files = [f for f in os.listdir(ckpt_dir) if f.startswith("step_") and f.endswith(".pt")]

    if not files:
        return None

    def extract_step(f):
        return int(re.findall(r"\d+", f)[0])

    latest_file = max(files, key=extract_step)
    return os.path.join(ckpt_dir, latest_file)

optimizer = AdamW(model.parameters(), lr=2e-5)

# 📍 NEW: try resume
resume_ckpt = get_latest_checkpoint(ckpt1)

start_step = 0

if resume_ckpt:
    print(f"🔄 Resuming from checkpoint: {resume_ckpt}")

    ckpt = torch.load(resume_ckpt, map_location=model.device)

    model.load_state_dict(ckpt["model"], strict=False)

    optimizer.load_state_dict(ckpt["optimizer"])

    for state in optimizer.state.values():
      for k, v in state.items():
          if torch.is_tensor(v):
              state[k] = v.to(model.device)

    start_step = ckpt["step"]

else:
    print("🆕 No checkpoint found, starting fresh")

🔄 Resuming from checkpoint: /content/drive/MyDrive/LegalAI/LegalAIDatasets/qwen_stage1_pretraining_ckpt/step_18000.pt


In [19]:
train_ds = Stage1Dataset("train")

train_loader = DataLoader(
    train_ds,
    batch_size=4,
    collate_fn=collate_stage1,
    num_workers=0,   # FIX: safer for IterableDataset streaming
    pin_memory=True
)

optimizer = AdamW(model.parameters(), lr=2e-5)

model.train()

accum_steps = 8
global_step = start_step

for epoch in range(2):

    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):

        input_ids = batch["input_ids"].to(model.device, non_blocking=True)
        labels = batch["labels"].to(model.device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(model.device, non_blocking=True)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        # ✅ FIX 1: correct gradient accumulation
        scaled_loss = loss / accum_steps
        scaled_loss.backward()

        # logging raw loss (not scaled)
        # if global_step % 50 == 0:
        #     print(f"[Stage1] Epoch {epoch} Step {global_step} Loss {loss.item():.4f}")

        # optimizer step
        if (step + 1) % accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

            global_step += 1

            # 🔥 LOG ONLY AFTER A REAL UPDATE
            if global_step % 50 == 0:
                print(
                    f"[Stage1] Epoch {epoch} "
                    f"Step {global_step} "
                    f"Loss {loss.item():.4f}"
                )

            if global_step % 1000 == 0 and global_step > 0:

                torch.save(
                    {
                        "step": global_step,
                        "model": model.state_dict(),
                        "optimizer": optimizer.state_dict()
                    },
                    f"{ckpt1}/step_{global_step}.pt"
                )

                print(f"💾 Saved step checkpoint: {global_step}")

    # ✅ FIX 2: flush leftover gradients at epoch end
    # if (step + 1) % accum_steps != 0:
    #     torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    #     optimizer.step()
    #     optimizer.zero_grad()

    # checkpoint save
    torch.save(
        {
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict()
        },
        f"{ckpt1}/epoch_{epoch}.pt"
    )

    print(f"✅ Saved checkpoint for epoch {epoch}")


== NOW PROCESSING CSV: 1.2df_txt_norm.csv ==



CheckpointError: torch.utils.checkpoint: A different number of tensors was saved during the original forward and recomputation.
Number of tensors saved during forward: 109
Number of tensors saved during recomputation: 26.

Tip: To see a more detailed error message, either pass `debug=True` to
`torch.utils.checkpoint.checkpoint(...)` or wrap the code block
with `with torch.utils.checkpoint.set_checkpoint_debug_enabled(True):` to
enable checkpoint‑debug mode globally.


# -------------------------STAGE 2 FINETUNING------------------------------

In [ ]:
def format_instruction(text):
    return f"""
### Instruction:
Explain the following legal text clearly

### Input:
{text}

### Response:
"""

In [ ]:
class Stage2Dataset(IterableDataset):
    def __init__(self, mode):
        self.mode = mode

    def __iter__(self):

        for text in unified_stream():

            if not split_filter(text, self.mode):
                continue

            prompt = format_instruction(text)
            full = prompt + text[:300]

            tokens = tokenizer.encode(full, truncation=True, max_length=1024)

            yield torch.tensor(tokens)

In [ ]:
def collate_stage2(batch):
    max_len = max(len(x) for x in batch)

    inputs, targets = [], []

    for x in batch:
        x = x.tolist() + [tokenizer.eos_token_id]

        padded = x + [tokenizer.eos_token_id] * (max_len - len(x))

        inp = torch.tensor(padded[:-1])
        tgt = torch.tensor(padded[1:])

        tgt[tgt == tokenizer.eos_token_id] = -100

        inputs.append(inp)
        targets.append(tgt)

    return torch.stack(inputs), torch.stack(targets)

#Load stage 1 model

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    f"{ckpt1}/epoch_2.pt",
    device_map="auto",
    torch_dtype=torch.float16
)

#CHECKPOINT STAGE 2

In [ ]:
ckpt2 = "/content/drive/MyDrive/qwen_stage2_ckpt"
os.makedirs(ckpt2, exist_ok=True)

#Train Stage 2

In [ ]:
train_ds = Stage2Dataset("train")
train_loader = DataLoader(train_ds, batch_size=4, collate_fn=collate_stage2)

optimizer = AdamW(model.parameters(), lr=1e-5)

model.train()

for epoch in range(3):

    for step, (x, y) in enumerate(train_loader):
        x, y = x.to(model.device), y.to(model.device)

        loss = model(input_ids=x, labels=y).loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if step % 50 == 0:
            print(f"[Stage2] Epoch {epoch} Step {step} Loss {loss.item()}")

    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict()
    }, f"{ckpt2}/epoch_{epoch}.pt")